In [4]:
import pandas as pd
import numpy as np
import requests
import joblib
import os
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. CẤU HÌNH HẰNG SỐ 
# ==========================================
LATITUDE = 21.0050  
LONGITUDE = 106.8333 
MODEL_PATH = 'ensemble_model_pm25.pkl'
RAW_DATA_PATH = 'data_dongmai_all_raw.csv'
OUTPUT_CSV = '30_mau_ngau_nhien_10Apr_16Apr.csv' # Đã đổi tên file

# Mốc thời gian (Lùi 2 ngày để lấy đà Rolling 24h)
FETCH_START_DATE = "2026-04-08" 
EVAL_START_DATE = "2026-04-10"
END_DATE = "2026-04-16" # Đã điều chỉnh đến ngày 16/04/2026

WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
BASE_MET_FEATURES = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_u', 'wind_v']

# ==========================================
# 2. HÀM TRUY XUẤT VÀ XỬ LÝ
# ==========================================
def fetch_historical_data(start_date, end_date):
    """Lấy dữ liệu API vệ tinh"""
    try:
        w_params = {"latitude": LATITUDE, "longitude": LONGITUDE, "start_date": start_date, "end_date": end_date, "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m", "timezone": "Asia/Bangkok"}
        df_weather = pd.DataFrame(requests.get(WEATHER_API_URL, params=w_params).json()['hourly'])
        
        aq_params = {"latitude": LATITUDE, "longitude": LONGITUDE, "start_date": start_date, "end_date": end_date, "hourly": "pm2_5,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide,ozone,aerosol_optical_depth", "timezone": "Asia/Bangkok"}
        df_aq = pd.DataFrame(requests.get(AIR_QUALITY_API_URL, params=aq_params).json()['hourly'])
        
        df_merged = pd.merge(df_weather, df_aq, on='time', how='inner').dropna()
        df_merged['time'] = pd.to_datetime(df_merged['time'])
        return df_merged
    except Exception as e:
        print(f"Lỗi API: {e}")
        return None

def extract_random_samples():
    if not os.path.exists(MODEL_PATH) or not os.path.exists(RAW_DATA_PATH):
        print("Lỗi: Thiếu file mô hình hoặc dữ liệu gốc.")
        return

    # 1. Tải mô hình và Feature Names
    model_data = joblib.load(MODEL_PATH)
    model = model_data['model']
    feature_names = model_data['feature_names']

    # 2. Khởi tạo lại Scaler chuẩn
    print("Đang chuẩn bị bộ quy đổi (Scaler) và tải dữ liệu...")
    df_base = pd.read_csv(RAW_DATA_PATH)
    df_base['time'] = pd.to_datetime(df_base['time'])
    for col in ['hour', 'day_of_week', 'month']:
        df_base[col] = getattr(df_base['time'].dt, col if col != 'day_of_week' else 'dayofweek')
    
    wind_dir_rad_base = np.radians(df_base['wind_direction_10m'])
    df_base['wind_u'] = -df_base['wind_speed_10m'] * np.sin(wind_dir_rad_base)
    df_base['wind_v'] = -df_base['wind_speed_10m'] * np.cos(wind_dir_rad_base)
    
    for feature in BASE_MET_FEATURES:
        df_base[f'{feature}_rolling_12h'] = df_base[feature].rolling(window=12, min_periods=1).mean()
        df_base[f'{feature}_rolling_24h'] = df_base[feature].rolling(window=24, min_periods=1).mean()
        
    scaler = MinMaxScaler().fit(df_base.dropna()[feature_names])

    # 3. Xử lý dữ liệu mục tiêu
    df_raw = fetch_historical_data(FETCH_START_DATE, END_DATE)
    if df_raw is None or df_raw.empty: return
    
    for col in ['hour', 'day_of_week', 'month']:
        df_raw[col] = getattr(df_raw['time'].dt, col if col != 'day_of_week' else 'dayofweek')
        
    wind_dir_rad = np.radians(df_raw['wind_direction_10m'])
    df_raw['wind_u'] = -df_raw['wind_speed_10m'] * np.sin(wind_dir_rad)
    df_raw['wind_v'] = -df_raw['wind_speed_10m'] * np.cos(wind_dir_rad)
    
    for feature in BASE_MET_FEATURES:
        df_raw[f'{feature}_rolling_12h'] = df_raw[feature].rolling(window=12, min_periods=1).mean()
        df_raw[f'{feature}_rolling_24h'] = df_raw[feature].rolling(window=24, min_periods=1).mean()
        
    # Lọc đúng từ ngày 10/04 đến 16/04
    df_eval = df_raw[df_raw['time'] >= pd.to_datetime(EVAL_START_DATE)].reset_index(drop=True)
    
    # 4. Chạy dự đoán AI
    df_features = df_eval[feature_names].copy()
    df_features[feature_names] = scaler.transform(df_features[feature_names])
    df_eval['AI_Predicted_PM25'] = model.predict(df_features).round(2)
    df_eval['pm2_5'] = df_eval['pm2_5'].round(2)
    
    # Tính độ lệch tuyệt đối
    df_eval['Absolute_Error'] = abs(df_eval['AI_Predicted_PM25'] - df_eval['pm2_5']).round(2)

    # 5. Lấy mẫu ngẫu nhiên 30 dòng và sắp xếp lại theo thời gian
    print("\nĐang bốc ngẫu nhiên 30 điểm thời gian từ 10/04 đến 16/04...")
    df_sampled = df_eval.sample(n=30, random_state=42) 
    
    df_sampled = df_sampled.sort_values(by='time').reset_index(drop=True)
    df_sampled['Formatted_Time'] = df_sampled['time'].dt.strftime('%H:%M - %d/%m/%Y')

    # Trích xuất các cột cần thiết
    final_cols = ['Formatted_Time', 'pm2_5', 'AI_Predicted_PM25', 'Absolute_Error']
    df_final = df_sampled[final_cols].copy()
    df_final.columns = ['Thời gian', 'Thực tế đo từ vệ tinh (μg/m3)', 'Dự đoán từ mô hình (μg/m3)', 'Độ lệch tuyệt đối (μg/m3)']
    df_final.index = df_final.index + 1 

    # Xuất file
    df_final.to_csv(OUTPUT_CSV, encoding='utf-8-sig')
    
    print("\n" + "="*70)
    print(" KẾT QUẢ 5 MẪU ĐẦU TIÊN (ĐÃ XUẤT RA FILE CSV)")
    print("="*70)
    print(df_final.head().to_string())
    print(f"\n[THÀNH CÔNG] Đã lưu đầy đủ 30 mẫu vào file: {OUTPUT_CSV}")
    
    mae_30_samples = df_final['Độ lệch tuyệt đối (μg/m3)'].mean()
    print(f"[*] MAE trên 30 mẫu ngẫu nhiên này là: {mae_30_samples:.2f} μg/m3")

if __name__ == "__main__":
    extract_random_samples()

Đang chuẩn bị bộ quy đổi (Scaler) và tải dữ liệu...

Đang bốc ngẫu nhiên 30 điểm thời gian từ 10/04 đến 16/04...

 KẾT QUẢ 5 MẪU ĐẦU TIÊN (ĐÃ XUẤT RA FILE CSV)
            Thời gian  Thực tế đo từ vệ tinh (μg/m3)  Dự đoán từ mô hình (μg/m3)  Độ lệch tuyệt đối (μg/m3)
1  09:00 - 10/04/2026                           43.8                       35.16                       8.64
2  12:00 - 10/04/2026                           58.6                       37.93                      20.67
3  15:00 - 10/04/2026                           62.4                       36.85                      25.55
4  16:00 - 10/04/2026                           58.6                       39.10                      19.50
5  18:00 - 10/04/2026                           44.2                       39.20                       5.00

[THÀNH CÔNG] Đã lưu đầy đủ 30 mẫu vào file: 30_mau_ngau_nhien_10Apr_16Apr.csv
[*] MAE trên 30 mẫu ngẫu nhiên này là: 8.84 μg/m3
